<a href="https://colab.research.google.com/github/ragabhumi/Praktikum_Magnetbumi/blob/main/Inversi_Data_Magnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Persiapan Lingkungan Python: Instalasi SimPEG & Visualisasi 3D

Langkah awal ini sangat krusial untuk memastikan semua pustaka komputasi dan visualisasi tersedia:
* **SimPEG (Simulation and Parameter Estimation in Geophysics)**: Engine utama yang menangani matematika inversi.
* **discretize**: Digunakan untuk mengelola mesh (grid) 3D dan operator kalkulus numerik.
* **PyVista & Trame**: Toolkit canggih untuk rendering visualisasi 3D interaktif langsung di dalam notebook.
* **nest_asyncio**: Diperlukan agar backend plotting interaktif dapat berjalan lancar di lingkungan asinkronus Colab.

*Catatan: Proses instalasi ini mungkin memakan waktu 1-2 menit.*

In [ ]:
!pip install simpeg discretize pymatsolver pyvista trame trame-vtk trame-vuetify trame-jupyter-extension nest_asyncio2 ipywidgets -q

# 2. Import Library dan Modul Utama

Kita membagi import menjadi beberapa bagian:
1. **Standar**: `numpy` untuk numerik dan `pandas` untuk manajemen tabel data.
2. **Visualisasi**: `matplotlib` untuk plot 2D dan `pyvista` untuk eksplorasi 3D.
3. **SimPEG Core**:
   - `maps`: Mengatur hubungan antara model inversi dan properti fisik sel.
   - `data_misfit`: Menghitung sejauh mana model kita cocok dengan observasi.
   - `regularization`: Memberikan batasan geologi (seperti kehalusan model) agar solusi menjadi stabil.
   - `optimization`: Algoritma (seperti Gauss-Newton) untuk mencari model terbaik.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import pyvista as pv

from discretize import TensorMesh
from discretize.utils import active_from_xyz

from simpeg import (
    maps,
    data,
    data_misfit,
    regularization,
    optimization,
    inverse_problem,
    inversion,
    directives,
)

from simpeg.potential_fields import magnetics
import nest_asyncio2
import ipywidgets as widgets
from IPython.display import display, clear_output

pv.set_jupyter_backend('html')
pv.global_theme.notebook = True

# 3. Akuisisi Data: Upload File Anomali Magnetik

Silakan unggah file data Anda. Pastikan data tersebut adalah **Total Magnetic Intensity (TMI)** yang sudah dikoreksi harian (diurnal) dan IGRF jika diperlukan.

Format yang didukung biasanya berupa kolom: `X | Y | Z | TMI | Noise`. Koordinat harus dalam sistem proyeksi meter (seperti UTM) agar perhitungan volume dan kedalaman menjadi akurat secara fisik.

In [ ]:
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

print("File uploaded:", file_name)

# 4. Pra-pemrosesan dan Analisis Statistik Data

Langkah ini melibatkan:
* **Pembersihan**: Menghapus baris kosong atau komentar.
* **Statistik**: Mengecek nilai minimum/maksimum untuk mendeteksi *spikes* (noise ekstrem) yang dapat merusak hasil inversi.
* **Koordinat**: Memastikan rentang survei (X dan Y) sudah sesuai untuk dasar pembuatan mesh di langkah berikutnya.

In [ ]:
df = pd.read_csv(
    file_name,
    sep=r"\s+|,",
    engine="python",
    comment="#",
    header=None
)

df.columns = ["x", "y", "z", "tmi", "noise"]

x = df["x"].values
y = df["y"].values
z = df["z"].values
dobs = df["tmi"].values
noise = df["noise"].values

receiver_locations = np.c_[x, y, z]

print(df.head())
print("Jumlah data:", len(dobs))
print("TMI min/max:", dobs.min(), dobs.max())
print("X range:", x.min(), x.max())
print("Y range:", y.min(), y.max())
print("Z range:", z.min(), z.max())

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(x, y, c=dobs, s=8, cmap=mpl.colormaps['hsv'])
plt.gca().set_aspect("equal")
plt.colorbar(label="Magnetic anomaly (nT)")
plt.xlabel("UTM X")
plt.ylabel("UTM Y")
plt.title("Observed magnetic anomaly")
plt.show()

# 5. Estimasi Ketidakpastian (Uncertainty Assignment)

Inversi tidak mencari kecocokan sempurna (misfit nol), melainkan kecocokan dalam rentang error data.

**Mengapa tidak memakai noise instrumen saja?**
Noise instrumen seringkali terlalu optimis (terlalu kecil). Kita menambahkan *percentage error* (misal 5%) untuk mengakomodasi error posisi atau kesalahan pemodelan, ditambah *noise floor* (misal 0.5 nT) agar data yang nilainya mendekati nol tidak mendominasi proses pembobotan.

In [ ]:
standard_deviation = 0.05 * np.abs(dobs) + 0.5

# 6. Konfigurasi Parameter Medan Magnetik Bumi

Karena anomali magnetik bersifat dipolar dan bergantung pada arah medan utama bumi, kita harus mendefinisikan:
* **Inclination & Declination**: Menentukan orientasi induksi pada batuan.
* **Field Strength**: Menentukan amplitudo respon.

Data ini bisa didapatkan dari nilai rata-rata survei atau dari kalkulator IGRF berdasarkan lokasi koordinat survei Anda.

In [ ]:
inclination = -31.069
declination = 1.001
field_strength = 44553.0

receiver_list = [
    magnetics.receivers.Point(
        receiver_locations,
        components=["tmi"]
    )
]

source_field = magnetics.sources.UniformBackgroundField(
    receiver_list=receiver_list,
    amplitude=field_strength,
    inclination=inclination,
    declination=declination,
)

survey = magnetics.survey.Survey(source_field)

Buat objek data:

In [ ]:
data_object = data.Data(
    survey,
    dobs=dobs,
    standard_deviation=standard_deviation
)

# 7. Diskritisasi: Perancangan Mesh 3D Tensor

Mesh adalah representasi volume bawah permukaan. Kita menggunakan strategi **Padding**:
1. **Core Region**: Sel berukuran kecil (misal 750m) di area data agar detail anomali tertangkap.
2. **Padding Region**: Sel yang semakin membesar di bagian pinggir dan bawah untuk menjauhkan efek batas (*boundary conditions*) tanpa menambah beban komputasi secara berlebihan.

In [ ]:
dh = 750.0      # ukuran sel horizontal, meter
dz = 750.0      # ukuran sel vertikal, meter
npad = 6         # padding
ncz = 25         # jumlah sel vertikal utama

Buat jumlah sel horizontal mengikuti ukuran data:

In [ ]:
ncx = int(np.ceil((x.max() - x.min()) / dh)) + 4
ncy = int(np.ceil((y.max() - y.min()) / dh)) + 4

hx = [(dh, npad, -1.3), (dh, ncx), (dh, npad, 1.3)]
hy = [(dh, npad, -1.3), (dh, ncy), (dh, npad, 1.3)]
hz = [(dz, npad, -1.3), (dz, ncz)]

mesh = TensorMesh([hx, hy, hz])

# Geser mesh supaya mencakup area UTM data
mesh.x0 = np.r_[
    x.mean() - mesh.h[0].sum() / 2,
    y.mean() - mesh.h[1].sum() / 2,
    -mesh.h[2].sum()
]

print(mesh)
print("Jumlah total cell:", mesh.nC)

# 8. Pemodelan Topografi dan Penentuan Sel Aktif

Kita mendefinisikan batas antara udara dan batuan. Sel-sel yang pusatnya berada di atas topografi akan ditandai sebagai 'tidak aktif' (suseptibilitas = 0) dan dikeluarkan dari proses perhitungan inversi untuk menghemat memori.

In [ ]:
topo_x, topo_y = np.meshgrid(
    np.linspace(mesh.nodes_x.min(), mesh.nodes_x.max(), 50),
    np.linspace(mesh.nodes_y.min(), mesh.nodes_y.max(), 50)
)

topo_z = np.zeros_like(topo_x)

topography = np.c_[
    topo_x.ravel(),
    topo_y.ravel(),
    topo_z.ravel()
]

Tentukan sel aktif di bawah permukaan:

In [ ]:
active_cells = active_from_xyz(mesh, topography)

n_active = int(active_cells.sum())
print("Jumlah active cells:", n_active)

# 9. Pemetaan Parameter Model (Mapping)

Mapping adalah jembatan antara dunia matematika dan dunia fisik:
* Model inversi berupa vektor numerik sederhana.
* Mapping mengubah vektor tersebut menjadi properti fisik (Suseptibilitas) yang terdistribusi secara spasial pada mesh 3D yang kita buat.

In [ ]:
active_map = maps.InjectActiveCells(
    mesh,
    active_cells,
    np.nan
)

model_map = active_map

Model awal:

In [ ]:
starting_model = np.ones(n_active) * 1e-4
reference_model = np.zeros(n_active)

# 10. Forward Modeling: Simulasi Respon Magnetik

Di sini kita membangun matriks sensitivitas ($G$). Matriks ini merepresentasikan bagaimana setiap sel di bawah tanah berkontribusi terhadap nilai yang terbaca di permukaan.

**Tips Efisiensi**: Karena matriks ini bisa berukuran sangat besar (Ribuan data x Ratusan ribu sel), kita menyimpannya di disk untuk mencegah Google Colab mengalami *out-of-memory* (OOM).

In [ ]:
simulation = magnetics.simulation.Simulation3DIntegral(
    mesh,
    survey=survey,
    chiMap=model_map,
    active_cells=active_cells,
    store_sensitivities="disk",
    sensitivity_path="./sensitivity"
)

# 11. Formulasi Fungsi Objektif Inversi

Kita mencari model ($m$) yang meminimalkan:
$$\Phi(m) = \Phi_d(m) + \beta \Phi_m(m)$$

*   **$\Phi_d$ (Data Misfit)**: Memastikan model sesuai dengan kenyataan lapangan.
*   **$\Phi_m$ (Regularization)**: Menjaga model tetap stabil dan halus secara spasial.
*   **$\beta$ (Trade-off parameter)**: Penyeimbang antara kecocokan data dan kemulusan model.

In [ ]:
dmis = data_misfit.L2DataMisfit(
    data=data_object,
    simulation=simulation
)

**Regularisasi**:

**Regularisasi** adalah proses untuk mengontrol bentuk model bawah permukaan agar hasil inversi tidak terlalu liar, tidak terlalu kasar, dan tetap masuk akal secara geologi.

Masalah inversi magnetik bersifat **non-unik**, artinya lebih dari satu model susceptibility dapat menghasilkan respons magnetik yang mirip. Oleh karena itu, inversi tidak cukup hanya mencari model yang paling cocok dengan data observasi, tetapi juga harus mencari model yang stabil dan realistis.

Secara umum, fungsi objektif dalam inversi dapat ditulis sebagai:

$$
\Phi = \Phi_d + \beta \Phi_m
$$

dengan:

- $\Phi_d$ = **data misfit**, yaitu ukuran perbedaan antara data observasi dan data prediksi.
- $\Phi_m$ = **model regularization**, yaitu ukuran kompleksitas atau kekasaran model.
- $\beta$ = parameter **trade-off** antara kecocokan data dan kesederhanaan model.

Jika nilai regularisasi terlalu kuat, model hasil inversi akan menjadi terlalu halus dan data prediksi tidak mampu mengikuti data observasi dengan baik. Kondisi ini disebut **underfit**.

Sebaliknya, jika regularisasi terlalu lemah, model dapat menjadi terlalu kasar atau terlalu mengikuti noise pada data. Kondisi ini disebut **overfit**.


In [ ]:
reg = regularization.WeightedLeastSquares(
    mesh,
    active_cells=active_cells,
    reference_model=reference_model
)

**Optimisasi**:

Optimisasi adalah proses mencari model terbaik dalam inversi dengan cara meminimalkan perbedaan antara data observasi dan data prediksi.

Dalam inversi magnetik 3D, optimisasi memperbarui model susceptibility secara bertahap pada setiap iterasi sampai data prediksi mendekati data observasi.

Secara sederhana:

optimisasi = proses mencari model yang menghasilkan misfit terkecil

In [ ]:
opt = optimization.ProjectedGNCG(
    maxIter=25,
    lower=0.0,
    upper=5.0,
    maxIterLS=20,
    maxIterCG=30,
    tolCG=1e-3
)

Batas lower=0.0 artinya susceptibility tidak boleh negatif. Untuk inversi induced magnetization biasa, ini masuk akal.

Gabungkan inverse problem:

In [ ]:
inv_prob = inverse_problem.BaseInvProblem(
    dmis,
    reg,
    opt
)

**Directives**:

Directives adalah instruksi tambahan di SimPEG yang mengatur jalannya proses inversi.

Directives tidak menghitung respons magnetik secara langsung, tetapi membantu mengontrol strategi inversi, misalnya:

- mengatur beta
- menurunkan beta secara bertahap
- menghitung sensitivity weighting
- memperbarui preconditioner
- menghentikan inversi saat target misfit tercapai

In [ ]:
directives_list = [
    directives.UpdateSensitivityWeights(every_iteration=False),
    directives.BetaEstimate_ByEig(beta0_ratio=1),
    directives.BetaSchedule(coolingFactor=2, coolingRate=1),
    directives.UpdatePreconditioner(),
    directives.TargetMisfit(chifact=1.0),
]

Buat inversion object:

In [ ]:
magnetic_inversion = inversion.BaseInversion(
    inv_prob,
    directiveList=directives_list
)

# 12. Eksekusi Iterasi Inversi

Proses ini adalah pencarian solusi secara bertahap. Pada setiap iterasi, algoritma akan memperbarui nilai suseptibilitas di setiap sel dan mengecek apakah 'misfit' sudah mencapai target (biasanya $\Phi_d \approx N_{data}$).

In [ ]:
recovered_model = magnetic_inversion.run(starting_model)

Target idealnya phi_d mendekati jumlah data:

In [ ]:
print("Jumlah data:", survey.nD)

# 13. Verifikasi Hasil: Analisis Fit dan Residual

Model yang bagus tidak hanya memberikan gambar yang indah, tapi harus bisa merekonstruksi data observasi.
* Jika **Residual** masih membentuk pola anomali yang jelas, berarti ada informasi di data yang belum terserap oleh model (Underfit).
* Jika **Residual** sangat acak (seperti semut), berarti model sudah bekerja optimal.

In [ ]:
dpred = simulation.dpred(recovered_model)
residual = dobs - dpred

print("Observed min/max:", dobs.min(), dobs.max())
print("Predicted min/max:", dpred.min(), dpred.max())
print("Residual min/max:", residual.min(), residual.max())

Cek metrik misfit.

Patokan sederhana:

phi_d / nD ≈ 1     : fit sesuai uncertainty

phi_d / nD >> 1    : underfit

phi_d / nD << 1    : overfit

Kalau Relative RMS residual masih besar, misalnya >0.3–0.5, fit-nya belum bagus.

In [ ]:
normalized_residual = residual / standard_deviation
phi_d = np.sum(normalized_residual**2)

print("Number of data:", survey.nD)
print("phi_d:", phi_d)
print("phi_d / nD:", phi_d / survey.nD)

rms_residual = np.sqrt(np.mean(residual**2))
rms_observed = np.sqrt(np.mean(dobs**2))

print("RMS observed:", rms_observed)
print("RMS residual:", rms_residual)
print("Relative RMS residual:", rms_residual / rms_observed)

Cek amplitudo model hasil inversi. Kalau maksimum susceptibility masih sangat kecil, misalnya:

0.001 atau 0.005 SI

maka model belum cukup kuat untuk menjelaskan data.

In [ ]:
print("Recovered susceptibility min/max:", recovered_model.min(), recovered_model.max())
print("Recovered susceptibility mean:", recovered_model.mean())

Plot perbandingan:

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(x, y, c=dpred, s=8, cmap=mpl.colormaps['hsv'])
plt.gca().set_aspect("equal")
plt.colorbar(label="Predicted magnetic anomaly (nT)")
plt.xlabel("UTM X")
plt.ylabel("UTM Y")
plt.title("Predicted data")
plt.show()

plt.figure(figsize=(7, 6))
plt.scatter(x, y, c=residual, s=8, cmap=mpl.colormaps['hsv'])
plt.gca().set_aspect("equal")
plt.colorbar(label="Residual (nT)")
plt.xlabel("UTM X")
plt.ylabel("UTM Y")
plt.title("Observed - Predicted")
plt.show()

# 14. Interpretasi Geologi: Slicing Model 3D

Setelah mendapatkan distribusi suseptibilitas, kita melakukan interpretasi:
* **High Susceptibility**: Bisa mengindikasikan keberadaan batuan beku, intrusi, atau mineralisasi besi.
* **Depth Analysis**: Kita bisa memperkirakan kedalaman puncak (*top*) dari badan anomali tersebut.

In [ ]:
recovered_model_full = active_map * recovered_model

Plot slice horizontal sederhana:

In [ ]:
# pilih slice kira-kira di kedalaman tertentu
z_centers = mesh.cell_centers[:, 2]
target_depth = -5000  # meter
ind_z = np.argmin(np.abs(mesh.cell_centers[:, 2] - target_depth))

print("Kedalaman slice terdekat:", mesh.cell_centers[ind_z, 2])

Untuk visualisasi lebih rapi, SimPEG/discretize punya fungsi plotting mesh. Coba:

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

mesh.plot_slice(
    recovered_model_full,
    normal="Z",
    ax=ax,
    ind=int(np.argmin(np.abs(mesh.cell_centers_z - target_depth))),
    grid=False,
    pcolor_opts={"cmap": "hsv"}
)

ax.set_title(f"Recovered susceptibility slice, z ≈ {target_depth} m")
ax.set_xlabel("UTM X")
ax.set_ylabel("UTM Y")
plt.show()

# 15. Eksplorasi Model 3D Interaktif

Visualisasi statis seringkali tidak cukup untuk memahami geometri kompleks badan bijih atau struktur bawah permukaan. Di bagian ini, kita menggunakan **PyVista** yang diintegrasikan dengan **Ipywidgets** untuk menciptakan dashboard kontrol interaktif:

### Fitur Utama Dashboard:
*   **Orthogonal Slicing**: Menggeser irisan pada sumbu X (Easting), Y (Northing), dan Z (Kedalaman) secara independen untuk melihat struktur dari berbagai sudut.
*   **Thresholding**: Menghilangkan sel dengan suseptibilitas rendah agar kita bisa fokus hanya pada 'inti' anomali (misalnya, hanya melihat batuan dengan suseptibilitas > 0.001 SI).
*   **Voxel Transparency**: Memberikan efek semi-transparan pada batuan sekitar agar posisi anomali terlihat jelas dalam ruang 3D.

*Gunakan slider di bawah ini untuk membedah model Anda secara dinamis.*

In [ ]:
recovered_model_full = active_map * recovered_model

susceptibility = np.asarray(recovered_model_full, dtype=float)

# Ganti NaN atau inf menjadi 0 agar aman divisualisasikan
susceptibility[~np.isfinite(susceptibility)] = 0.0

vtk_model = pv.RectilinearGrid(
    mesh.nodes_x,
    mesh.nodes_y,
    mesh.nodes_z
)

vtk_model.cell_data["susceptibility"] = susceptibility

print("sus min:", susceptibility.min())
print("sus max:", susceptibility.max())

Buat fungsi plot

In [ ]:
def show_slice(axis="z", position=-5000, threshold_value=0.001):
    clear_output(wait=True)

    model_threshold = vtk_model.threshold(
        value=threshold_value,
        scalars="susceptibility"
    )

    if axis == "x":
        normal = "x"
        origin = (position, 0, 0)
    elif axis == "y":
        normal = "y"
        origin = (0, position, 0)
    else:
        normal = "z"
        origin = (0, 0, position)

    slice_model = vtk_model.slice(
        normal=normal,
        origin=origin
    )

    p = pv.Plotter(notebook=True)

    p.add_mesh(
        model_threshold,
        scalars="susceptibility",
        opacity=0.15,
        show_edges=False,
        clim=[sus_min, sus_max],
    )

    p.add_mesh(
        slice_model,
        scalars="susceptibility",
        opacity=1.0,
        clim=[sus_min, sus_max],
    )

    p.add_axes()
    p.show_grid()
    p.show()

def show_slice(axis="z", position=-5000, threshold_value=0.001):
    clear_output(wait=True)

    model_threshold = vtk_model.threshold(
        value=threshold_value,
        scalars="susceptibility"
    )

    if axis == "x":
        normal = "x"
        origin = (position, 0, 0)
    elif axis == "y":
        normal = "y"
        origin = (0, position, 0)
    else:
        normal = "z"
        origin = (0, 0, position)

    slice_model = vtk_model.slice(
        normal=normal,
        origin=origin
    )

    p = pv.Plotter(notebook=True)

    p.add_mesh(
        model_threshold,
        scalars="susceptibility",
        opacity=0.15,
        show_edges=False,
        clim=[sus_min, sus_max],
    )

    p.add_mesh(
        slice_model,
        scalars="susceptibility",
        opacity=1.0,
        clim=[sus_min, sus_max],
    )

    p.add_axes()
    p.show_grid()
    p.show()

def show_orthogonal_slices(x_pos=None, y_pos=None, z_pos=None, threshold_value=0.001):
    clear_output(wait=True)

    if x_pos is None:
        x_pos = 0.5 * (x_min + x_max)
    if y_pos is None:
        y_pos = 0.5 * (y_min + y_max)
    if z_pos is None:
        z_pos = 0.5 * (z_min + z_max)

    model_threshold = vtk_model.threshold(
        value=threshold_value,
        scalars="susceptibility"
    )

    sx = vtk_model.slice(normal="x", origin=(x_pos, 0, 0))
    sy = vtk_model.slice(normal="y", origin=(0, y_pos, 0))
    sz = vtk_model.slice(normal="z", origin=(0, 0, z_pos))

    p = pv.Plotter(notebook=True)

    p.add_mesh(
        model_threshold,
        scalars="susceptibility",
        opacity=0.12,
        show_edges=False,
        clim=[sus_min, sus_max],
    )

    p.add_mesh(sx, scalars="susceptibility", clim=[sus_min, sus_max])
    p.add_mesh(sy, scalars="susceptibility", clim=[sus_min, sus_max])
    p.add_mesh(sz, scalars="susceptibility", clim=[sus_min, sus_max])

    p.add_axes()
    p.show_grid()
    p.show()

In [ ]:
out = widgets.Output()

x_slider = widgets.FloatSlider(
    value=0.5 * (vtk_model.bounds[0] + vtk_model.bounds[1]),
    min=vtk_model.bounds[0],
    max=vtk_model.bounds[1],
    step=1000,
    description="UTM X",
    continuous_update=False
)

y_slider = widgets.FloatSlider(
    value=0.5 * (vtk_model.bounds[2] + vtk_model.bounds[3]),
    min=vtk_model.bounds[2],
    max=vtk_model.bounds[3],
    step=1000,
    description="UTM Y",
    continuous_update=False
)

z_slider = widgets.FloatSlider(
    value=0.5 * (vtk_model.bounds[4] + vtk_model.bounds[5]),
    min=vtk_model.bounds[4],
    max=vtk_model.bounds[5],
    step=500,
    description="Depth/Z",
    continuous_update=False
)

threshold_slider = widgets.FloatSlider(
    value=0.001,
    min=0.0,
    max=0.005,
    step=0.0001,
    description="Threshold",
    continuous_update=False
)

def update_orthogonal_plot(x_pos, y_pos, z_pos, threshold_value):
    with out:
        clear_output(wait=True)

        model_threshold = vtk_model.threshold(
            value=threshold_value,
            scalars="susceptibility",
            preference="cell"
        )

        sx = vtk_model.slice(normal="x", origin=(x_pos, 0, 0))
        sy = vtk_model.slice(normal="y", origin=(0, y_pos, 0))
        sz = vtk_model.slice(normal="z", origin=(0, 0, z_pos))

        p = pv.Plotter(notebook=True)

        p.add_mesh(
            model_threshold,
            scalars="susceptibility",
            preference="cell",
            cmap="hsv",
            opacity=0.15,
            show_edges=False,
            clim=[0, 0.005],
            scalar_bar_args={
                "title": "Susceptibility (SI)"
            }
        )

        p.add_mesh(
            sx,
            scalars="susceptibility",
            preference="cell",
            cmap="hsv",
            clim=[0, 0.005],
        )

        p.add_mesh(
            sy,
            scalars="susceptibility",
            preference="cell",
            cmap="hsv",
            clim=[0, 0.005],
        )

        p.add_mesh(
            sz,
            scalars="susceptibility",
            preference="cell",
            cmap="hsv",
            clim=[0, 0.005],
        )

        p.add_axes()

        p.show_bounds(
            grid="back",
            location="outer",
            xlabel="UTM X",
            ylabel="UTM Y",
            zlabel="Depth (meter)"
        )

        p.show()

widgets.interactive_output(
    update_orthogonal_plot,
    {
        "x_pos": x_slider,
        "y_pos": y_slider,
        "z_pos": z_slider,
        "threshold_value": threshold_slider,
    }
)

display(x_slider, y_slider, z_slider, threshold_slider, out)

update_orthogonal_plot(
    x_slider.value,
    y_slider.value,
    z_slider.value,
    threshold_slider.value
)

# 16. Ekspor Data untuk Software Interpretasi Lanjut

Langkah terakhir adalah menyimpan model dalam format koordinat XYZ. File ini sangat kompatibel untuk diimpor ke software seperti **Oasis montaj**, **Leapfrog**, atau **Petrel** untuk digabungkan dengan data geologi atau seismik.

In [ ]:
np.savetxt(
    "recovered_susceptibility_active_cells.txt",
    recovered_model,
    header="susceptibility_SI"
)

Simpan cell center aktif + susceptibility:

In [ ]:
active_centers = mesh.cell_centers[active_cells]

output = np.c_[
    active_centers[:, 0],
    active_centers[:, 1],
    active_centers[:, 2],
    recovered_model
]

np.savetxt(
    "recovered_susceptibility_xyz.txt",
    output,
    header="x y z susceptibility_SI",
    fmt='%.8f'
)

Download dari Colab:

In [ ]:
from google.colab import files

files.download("recovered_susceptibility_xyz.txt")